This notebook prepares input for "10. Multiome 6 - Fig. 4A-D, S6A-G".

## Setup

In [1]:
renv::load('/oak/stanford/groups/agitler/Shared/Shared_Jupyter_Notebook_Analysis/4.1.1-OG-Multiome/')

library(ArchR)
library(future)
library(dplyr)
library(BSgenome.Mmusculus.UCSC.mm10)
library(pheatmap)
library(ggpubr)
library(ggrepel)
library(ggbreak)
library(cowplot)
library(Seurat)
library(forcats)

if(!dir.exists('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')){
  dir.create('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')
}
setwd('/oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome')

addArchRGenome("mm10")

plan(strategy = "multicore", workers = 16)
options(future.globals.maxSize = 41953040000)

addArchRThreads(threads = 16)


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

## Call alpha motor neuron peaks (control vs. mid/end)

In [2]:
#Load ArchR Project
proj_alpha <- loadArchRProject('proj_alpha')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [3]:
# Get metadata from the project
metadata_df <- getCellColData(proj_alpha, select = c("Sample", "Stage")) %>% as.data.frame()

# Summarize the number of nuclei for each sample/stage
summary_df_sample_stage <- metadata_df %>%
  group_by(Sample, Stage) %>%
  summarise(Num_Nuclei = n())

summary_df_stage <- metadata_df %>%
  group_by(Stage) %>%
  summarise(Num_Nuclei = n())

summary_df_sample_stage
summary_df_stage

`summarise()` has grouped output by 'Sample'. You can override using the `.groups` argument.


Sample,Stage,Num_Nuclei
<chr>,<chr>,<int>
sample1_12_22,control,171
sample1_12_22,early,365
sample1_5_25,control,446
sample1_5_25,mid-late,613
sample1_6_3,mid-late,669
sample2_12_22,control,435
sample2_12_22,early,192
sample2_6_3,mid-late,542


Stage,Num_Nuclei
<chr>,<int>
control,1052
early,557
mid-late,1824


In [4]:
#Call Peaks w/ Macs2
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [5]:
proj_alpha <- addGroupCoverages(
    ArchRProj = proj_alpha, 
    groupBy = "Stage"
    )
         
proj_alpha <- addReproduciblePeakSet(
    ArchRProj = proj_alpha, 
    groupBy = "Stage", 
    pathToMacs2 = pathToMacs2
    )
    
proj_alpha <- addPeakMatrix(proj_alpha)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-2a23bb1d1d5-Date-2024-04-24_Time-21-47-56.log
If there is an issue, please report to github with logFile!

control (1 of 3) : CellGroups N = 3

early (2 of 3) : CellGroups N = 2

mid-late (3 of 3) : CellGroups N = 3

2024-04-24 21:48:39 : Creating Coverage Files!, 0.729 mins elapsed.

2024-04-24 21:48:39 : Batch Execution w/ safelapply!, 0.729 mins elapsed.

2024-04-24 21:49:18 : Adding Kmer Bias to Coverage Files!, 1.369 mins elapsed.

Completed Kmer Bias Calculation

Adding Kmer Bias (1 of 8)

Adding Kmer Bias (2 of 8)

Adding Kmer Bias (3 of 8)

Adding Kmer Bias (4 of 8)

Adding Kmer Bias (5 of 8)

Adding Kmer Bias (6 of 8)

Adding Kmer Bias (7 of 8)

Adding Kmer Bias (8 of 8)

2024-04-24 21:49:56 : Finished Creation of Coverage Files!, 1.996 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGroupCoverages-2a23bb1d1d5-Date-2024-04-24_Time-21-47-56.log

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-2a233e9

            Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
control   control   1052       1052           3  171  446   150000
early       early    557        557           2  192  365   150000
mid-late mid-late   1824       1500           3  500  500   150000


2024-04-24 21:49:56 : Batching Peak Calls!, 0.007 mins elapsed.

2024-04-24 21:49:56 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-04-24 21:53:31 : Identifying Reproducible Peaks!, 3.578 mins elapsed.

2024-04-24 21:53:43 : Creating Union Peak Set!, 3.788 mins elapsed.

Converged after 5 iterations!

Plotting Ggplot!

2024-04-24 21:53:51 : Finished Creating Union Peak Set (190439)!, 3.914 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-2a2322fdbf1d-Date-2024-04-24_Time-21-53-51.log
If there is an issue, please report to github with logFile!

2024-04-24 21:53:51 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-2a2322fdbf1d-Date-2024-04-24_Time-21-53-51.log



In [6]:
#Save ArchR Project
saveArchRProject(ArchRProj = proj_alpha, outputDirectory = "proj_alpha")

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
         

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_alpha 
samples(5): sample2_6_3 sample1_5_25 sample2_12_22 sample1_6_3
  sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 3433
medianTSS(1): 8.747
medianFrags(1): 25770

### Positive TF regulator input by DAMN/DM-status

In [2]:
#Load ArchR Project
proj_alpha <- loadArchRProject('proj_alpha')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [3]:
#Add DAMN status annotations
newLabels <- c('Late DAMN', ' Non-DAMN', ' Non-DAMN',  
               'Early DAMN', ' Non-DAMN')

oldLabels <- c('Late DAMN', 'Intermediate', 'Fast-Firing', 'Early DAMN', 'Slow-Firing')

proj_alpha$DAMN_status <- mapLabels(proj_alpha$alpha_subtype, newLabels = newLabels, oldLabels = oldLabels)

In [4]:
#Identify Deviant TF Motifs
seGroupMotif <- getGroupSE(ArchRProj = proj_alpha, useMatrix = "MotifMatrix", groupBy = "DAMN_status")

ArchR logging to : ArchRLogs/ArchR-getGroupSE-fc37afc4130-Date-2026-06-22_Time-03-35-17.log
If there is an issue, please report to github with logFile!

Getting Group Matrix

2026-06-22 03:35:21 : Successfully Created Group Matrix, 0.043 mins elapsed.

Normalizing by number of Cells

ArchR logging successful to : ArchRLogs/ArchR-getGroupSE-fc37afc4130-Date-2026-06-22_Time-03-35-17.log



In [5]:
seZ <- seGroupMotif[rowData(seGroupMotif)$seqnames=="z",]

In [6]:
rowData(seZ)$maxDelta <- lapply(seq_len(ncol(seZ)), function(x){
  rowMaxs(assay(seZ) - assay(seZ)[,x])
}) %>% Reduce("cbind", .) %>% rowMaxs

In [7]:
#Identify Correlated TF Motifs and TF Gene Score/Expression
corGEM_MM <- correlateMatrices(
    ArchRProj = proj_alpha,
    useMatrix1 = "GeneExpressionMatrix",
    useMatrix2 = "MotifMatrix",
    reducedDims = "LSI_Combined"
)

ArchR logging to : ArchRLogs/ArchR-correlateMatrices-fc3113d9167-Date-2026-06-22_Time-03-35-23.log
If there is an issue, please report to github with logFile!

When accessing features from a matrix of class Sparse.Assays.Matrix it requires 1 seqname!
Continuing with first seqname 'z'!
If confused, try getFeatures(ArchRProj, 'MotifMatrix') to list out available seqnames for input!

2026-06-22 03:35:29 : Testing 797 Mappings!, 0.105 mins elapsed.

2026-06-22 03:35:29 : Computing KNN, 0.105 mins elapsed.

2026-06-22 03:35:29 : Identifying Non-Overlapping KNN pairs, 0.11 mins elapsed.

2026-06-22 03:35:31 : Identified 499 Groupings!, 0.135 mins elapsed.

2026-06-22 03:35:34 : Getting Group Matrix 1, 0.185 mins elapsed.

2026-06-22 03:35:40 : Getting Group Matrix 2, 0.297 mins elapsed.

Some entries in groupMat2 are less than 0, continuing without Log2 Normalization.
Most likely this assay is a deviations matrix.

Getting Correlations...

2026-06-22 03:35:44 : 

Computing Correlation (250 o

In [8]:
#Add Maximum Delta Deviation to the Correlation Data Frame
corGEM_MM$maxDelta <- rowData(seZ)[match(corGEM_MM$MotifMatrix_name, rowData(seZ)$name), "maxDelta"]

In [9]:
#Identify Positive TF Regulators
corGEM_MM <- corGEM_MM[order(abs(corGEM_MM$cor), decreasing = TRUE), ]
corGEM_MM <- corGEM_MM[which(!duplicated(gsub("\\-.*","",corGEM_MM[,"MotifMatrix_name"]))), ]
corGEM_MM$TFRegulator <- "NO"
corGEM_MM$TFRegulator[which(corGEM_MM$cor > 0.5 & corGEM_MM$padj < 0.01 & corGEM_MM$maxDelta > quantile(corGEM_MM$maxDelta, 0.75))] <- "YES"
sort(corGEM_MM[corGEM_MM$TFRegulator=="YES",1])

[1] "Ar"      "Atf1"    "Atf3"    "Atf5"    "Atf6"    "Bach1"   "Bhlhe40"
 [8] "Bhlhe41" "Cebpg"   "Creb3"   "Crem"    "E4f1"    "Esrra"   "Esrrb"  
[15] "Esrrg"   "Fosb"    "Fosl1"   "Fosl2"   "Gmeb2"   "Hbp1"    "Hoxc8"  
[22] "Jun"     "Junb"    "Jund"    "Klf13"   "Klf4"    "Klf6"    "Klf7"   
[29] "Nfatc2"  "Nfatc3"  "Nfe2l1"  "Nfe2l2"  "Nfe2l3"  "Nfia"    "Nfil3"  
[36] "Nfil3"   "Nr2f6"   "Nr3c1"   "Nr4a1"   "Sox5"    "Tcf4"    "Xbp1"

In [10]:
write.csv(corGEM_MM, file='proj_alpha_corGEM_MM.csv')

## Subset control skeletal motor neurons 

In [2]:
# Load ArchR project
proj_cholinergic_ctl <- loadArchRProject('proj_cholinergic_ctl')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [3]:
proj_skeletal_ctl <- proj_cholinergic_ctl[proj_cholinergic_ctl$cholinergic_type %in% c('Alpha MNs', 'Gamma MNs', 'Gamma* MNs')]

In [4]:
#Add skeletal type annotations (Alpha and Pan-Gamma MNs)
newLabels <- c('Alpha MNs', 'Pan-Gamma MNs', 'Pan-Gamma MNs')

oldLabels <- c('Alpha MNs', 'Gamma MNs', 'Gamma* MNs')

proj_skeletal_ctl$skeletal_type <- mapLabels(proj_skeletal_ctl$cholinergic_type, newLabels = newLabels, oldLabels = oldLabels)

In [5]:
#Save ArchR Project
saveArchRProject(ArchRProj = proj_skeletal_ctl, outputDirectory = "proj_skeletal_ctl")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_skeletal_ctl

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 8): Annotations

Copying Other Files (2 of 8): Background-Peaks.rds

Copying Other Files (3 of 8): Embeddings

Copying Other Files (4 of 8): GroupCoverages

Copying Other Files (5 of 8): LSI_ATAC

Copying Other Files (6 of 8): LSI_RNA

Copying Other Files (7 of 8): PeakCalls

Copying Other Files (8 of 8): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\           

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_skeletal_ctl 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(24): Sample TSSEnrichment ...
  cholinergic_type_UMAP_labels skeletal_type
numberOfCells(1): 1626
medianTSS(1): 9.829
medianFrags(1): 17182.5

### Call peaks

In [6]:
#Load ArchR Project
proj_skeletal_ctl <- loadArchRProject('proj_skeletal_ctl')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [7]:
#Call Peaks w/ Macs2
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [8]:
proj_skeletal_ctl <- addGroupCoverages(
    ArchRProj = proj_skeletal_ctl, 
    groupBy = "skeletal_type"
    )
         
proj_skeletal_ctl <- addReproduciblePeakSet(
    ArchRProj = proj_skeletal_ctl, 
    groupBy = "skeletal_type", 
    pathToMacs2 = pathToMacs2
    )
    
proj_skeletal_ctl <- addPeakMatrix(proj_skeletal_ctl)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-67a06f975a87-Date-2024-07-11_Time-09-43-07.log
If there is an issue, please report to github with logFile!

Alpha MNs (1 of 2) : CellGroups N = 3

Pan-Gamma MNs (2 of 2) : CellGroups N = 3

2024-07-11 09:43:18 : Creating Coverage Files!, 0.182 mins elapsed.

2024-07-11 09:43:18 : Batch Execution w/ safelapply!, 0.182 mins elapsed.

2024-07-11 09:43:44 : Adding Kmer Bias to Coverage Files!, 0.616 mins elapsed.

Completed Kmer Bias Calculation

Adding Kmer Bias (1 of 6)

Adding Kmer Bias (2 of 6)

Adding Kmer Bias (3 of 6)

Adding Kmer Bias (4 of 6)

Adding Kmer Bias (5 of 6)

Adding Kmer Bias (6 of 6)

2024-07-11 09:44:05 : Finished Creation of Coverage Files!, 0.959 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGroupCoverages-67a06f975a87-Date-2024-07-11_Time-09-43-07.log

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-67a0a03aee3-Date-2024-07-11_Time-09-44-05.log
If there is an issue, please report to gi

                      Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
Alpha MNs         Alpha MNs   1052       1052           3  171  446   150000
Pan-Gamma MNs Pan-Gamma MNs    574        574           3   89  294   150000


2024-07-11 09:44:05 : Batching Peak Calls!, 0.007 mins elapsed.

2024-07-11 09:44:05 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-07-11 09:45:54 : Identifying Reproducible Peaks!, 1.82 mins elapsed.

2024-07-11 09:51:32 : Creating Union Peak Set!, 7.449 mins elapsed.

Converged after 4 iterations!

Plotting Ggplot!

2024-07-11 09:51:40 : Finished Creating Union Peak Set (167445)!, 7.578 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-67a0346c87d6-Date-2024-07-11_Time-09-51-40.log
If there is an issue, please report to github with logFile!

2024-07-11 09:51:40 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-67a0346c87d6-Date-2024-07-11_Time-09-51-40.log



### Postive TF regulator input by skeletal MN subtype

In [9]:
# Add motif annotations
proj_skeletal_ctl <- addMotifAnnotations(ArchRProj = proj_skeletal_ctl, motifSet = "cisbp", name = "Motif", force = TRUE)

ArchR logging to : ArchRLogs/ArchR-addMotifAnnotations-67a042a9a7f4-Date-2024-07-11_Time-09-52-36.log
If there is an issue, please report to github with logFile!

2024-07-11 09:52:48 : Gettting Motif Set, Species : Mus musculus, 0.005 mins elapsed.

Using version 2 motifs!

2024-07-11 09:52:52 : Finding Motif Positions with motifmatchr!, 0.064 mins elapsed.

2024-07-11 09:55:33 : All Motifs Overlap at least 1 peak!, 2.755 mins elapsed.

2024-07-11 09:55:33 : Creating Motif Overlap Matrix, 2.755 mins elapsed.

2024-07-11 09:55:36 : Finished Getting Motif Info!, 2.793 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addMotifAnnotations-67a042a9a7f4-Date-2024-07-11_Time-09-52-36.log



In [10]:
# Add background peaks
proj_skeletal_ctl <- addBgdPeaks(proj_skeletal_ctl)

Identifying Background Peaks!



In [11]:
# Add deviations matrix
proj_skeletal_ctl <- addDeviationsMatrix(
  ArchRProj = proj_skeletal_ctl, 
  peakAnnotation = "Motif",
  force = TRUE
)

Using Previous Background Peaks!

ArchR logging to : ArchRLogs/ArchR-addDeviationsMatrix-67a0c7184cb-Date-2024-07-11_Time-09-55-51.log
If there is an issue, please report to github with logFile!



NULL


'as(<lgCMatrix>, "dgCMatrix")' is deprecated.
Use 'as(., "dMatrix")' instead.
See help("Deprecated") and help("Matrix-deprecated").

2024-07-11 09:55:54 : Batch Execution w/ safelapply!, 0 mins elapsed.

###########
2024-07-11 09:59:23 : Completed Computing Deviations!, 3.522 mins elapsed.
###########

ArchR logging successful to : ArchRLogs/ArchR-addDeviationsMatrix-67a0c7184cb-Date-2024-07-11_Time-09-55-51.log



In [3]:
#Identify Deviant TF Motifs
seGroupMotif <- getGroupSE(ArchRProj = proj_skeletal_ctl, useMatrix = "MotifMatrix", groupBy = "skeletal_type")

ArchR logging to : ArchRLogs/ArchR-getGroupSE-696fbc7174b-Date-2024-08-09_Time-14-47-24.log
If there is an issue, please report to github with logFile!

Getting Group Matrix

2024-08-09 14:47:44 : Successfully Created Group Matrix, 0.042 mins elapsed.

Normalizing by number of Cells

ArchR logging successful to : ArchRLogs/ArchR-getGroupSE-696fbc7174b-Date-2024-08-09_Time-14-47-24.log



In [4]:
seZ <- seGroupMotif[rowData(seGroupMotif)$seqnames=="z",]

In [5]:
rowData(seZ)$maxDelta <- lapply(seq_len(ncol(seZ)), function(x){
  rowMaxs(assay(seZ) - assay(seZ)[,x])
}) %>% Reduce("cbind", .) %>% rowMaxs

In [6]:
#Identify Correlated TF Motifs and TF Gene Score/Expression
corGEM_MM <- correlateMatrices(
    ArchRProj = proj_skeletal_ctl,
    useMatrix1 = "GeneExpressionMatrix",
    useMatrix2 = "MotifMatrix",
    reducedDims = "LSI_Combined"
)

ArchR logging to : ArchRLogs/ArchR-correlateMatrices-696fa00af64-Date-2024-08-09_Time-14-47-45.log
If there is an issue, please report to github with logFile!

When accessing features from a matrix of class Sparse.Assays.Matrix it requires 1 seqname!
Continuing with first seqname 'z'!
If confused, try getFeatures(ArchRProj, 'MotifMatrix') to list out available seqnames for input!

2024-08-09 14:47:57 : Testing 797 Mappings!, 0.204 mins elapsed.

2024-08-09 14:47:57 : Computing KNN, 0.204 mins elapsed.

2024-08-09 14:47:58 : Identifying Non-Overlapping KNN pairs, 0.218 mins elapsed.

2024-08-09 14:47:59 : Identified 500 Groupings!, 0.243 mins elapsed.

2024-08-09 14:48:02 : Getting Group Matrix 1, 0.281 mins elapsed.

2024-08-09 14:48:07 : Getting Group Matrix 2, 0.364 mins elapsed.

Some entries in groupMat2 are less than 0, continuing without Log2 Normalization.
Most likely this assay is a deviations matrix.

Getting Correlations...

2024-08-09 14:48:09 : 

Computing Correlation (250 

In [7]:
#Add Maximum Delta Deviation to the Correlation Data Frame
corGEM_MM$maxDelta <- rowData(seZ)[match(corGEM_MM$MotifMatrix_name, rowData(seZ)$name), "maxDelta"]

In [8]:
#Identify Positive TF Regulators
corGEM_MM <- corGEM_MM[order(abs(corGEM_MM$cor), decreasing = TRUE), ]
corGEM_MM <- corGEM_MM[which(!duplicated(gsub("\\-.*","",corGEM_MM[,"MotifMatrix_name"]))), ]
corGEM_MM$TFRegulator <- "NO"
corGEM_MM$TFRegulator[which(corGEM_MM$cor > 0.5 & corGEM_MM$padj < 0.01 & corGEM_MM$maxDelta > quantile(corGEM_MM$maxDelta, 0.75))] <- "YES"
sort(corGEM_MM[corGEM_MM$TFRegulator=="YES",1])

[1] "Ar"      "Arnt2"   "Arntl"   "Atf7"    "Ehf"     "Elk4"    "Esrrb"  
 [8] "Esrrg"   "Ets1"    "Gmeb1"   "Isl2"    "Jdp2"    "Klf1"    "Klf11"  
[15] "Klf12"   "Klf13"   "Klf5"    "Maf"     "Mbtps2"  "Mitf"    "Nfia"   
[22] "Nfib"    "Nr2c2"   "Nr3c1"   "Pbx3"    "Pou3f3"  "Rest"    "Scx"    
[29] "Smarcc2" "Sp1"     "Sp4"     "Stat3"   "Tef"     "Usf2"

In [9]:
corGEM_MM

DataFrame with 795 rows and 16 columns
    GeneExpressionMatrix_name MotifMatrix_name       cor         padj
                  <character>      <character> <numeric>    <numeric>
1                       Arnt2         Arnt2_26  0.965468 2.76938e-290
2                        Nfib         Nfib_859  0.960579 3.11672e-276
3                         Maf          Maf_793  0.947226 2.03047e-245
4                       Npas3         Npas3_34  0.939634 2.64315e-231
5                       Npas1         Npas1_19  0.929605 3.07282e-215
...                       ...              ...       ...          ...
791                      Zic3         Zic3_229        NA           NA
792                      Sox3         Sox3_739        NA           NA
793                       Arx          Arx_500        NA           NA
794                      Cdx4         Cdx4_481        NA           NA
795                     Tbx22        Tbx22_865        NA           NA
            pval GeneExpressionMatrix_seqnames Gene

In [11]:
write.csv(corGEM_MM, file='proj_skeletal_ctl_corGEM_MM.csv')

## Subset control alpha motor neurons

In [47]:
#Load ArchR Project
proj_alpha_control <- loadArchRProject('proj_alpha_control')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [48]:
proj_control_FF_Int_SF <- proj_alpha_control[proj_alpha_control$alpha_subtype %in% c('Fast-Firing', 'Intermediate', 'Slow-Firing')]

In [49]:
#Save ArchR Project
saveArchRProject(ArchRProj = proj_control_FF_Int_SF, outputDirectory = "proj_control_FF_Int_SF")

Copying ArchRProject to new outputDirectory : /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_control_FF_Int_SF

Copying Arrow Files...

Copying Arrow Files (1 of 3)

Copying Arrow Files (2 of 3)

Copying Arrow Files (3 of 3)

Getting ImputeWeights

No imputeWeights found, returning NULL

Copying Other Files...

Copying Other Files (1 of 9): Annotations

Copying Other Files (2 of 9): Background-Peaks.rds

Copying Other Files (3 of 9): Embeddings

Copying Other Files (4 of 9): GroupCoverages

Copying Other Files (5 of 9): LSI_ATAC

Copying Other Files (6 of 9): LSI_RNA

Copying Other Files (7 of 9): Peak2GeneLinks

Copying Other Files (8 of 9): PeakCalls

Copying Other Files (9 of 9): Plots

Saving ArchRProject...

Loading ArchRProject...

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                   

class: ArchRProject 
outputDirectory: /oak/stanford/groups/agitler/Shared/SOD1_Paper/Multiome/proj_control_FF_Int_SF 
samples(3): sample1_5_25 sample2_12_22 sample1_12_22
sampleColData names(1): ArrowFiles
cellColData names(26): Sample TSSEnrichment ... ReadsInPeaks FRIP
numberOfCells(1): 1018
medianTSS(1): 8.5665
medianFrags(1): 23968.5

### Call peaks

In [50]:
#Load ArchR Project
proj_control_FF_Int_SF <- loadArchRProject('proj_control_FF_Int_SF')

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [51]:
#Call Peaks w/ Macs2
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [52]:
proj_control_FF_Int_SF <- addGroupCoverages(
    ArchRProj = proj_control_FF_Int_SF, 
    groupBy = "alpha_subtype"
    )
         
proj_control_FF_Int_SF <- addReproduciblePeakSet(
    ArchRProj = proj_control_FF_Int_SF, 
    groupBy = "alpha_subtype", 
    pathToMacs2 = pathToMacs2
    )
    
proj_control_FF_Int_SF <- addPeakMatrix(proj_control_FF_Int_SF)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-53d3f42c958-Date-2024-05-05_Time-21-19-40.log
If there is an issue, please report to github with logFile!

Fast-Firing (1 of 3) : CellGroups N = 3

Intermediate (2 of 3) : CellGroups N = 2

Slow-Firing (3 of 3) : CellGroups N = 3

2024-05-05 21:19:46 : Creating Coverage Files!, 0.094 mins elapsed.

2024-05-05 21:19:46 : Batch Execution w/ safelapply!, 0.094 mins elapsed.

2024-05-05 21:20:24 : Adding Kmer Bias to Coverage Files!, 0.724 mins elapsed.

Completed Kmer Bias Calculation

Adding Kmer Bias (1 of 8)

Adding Kmer Bias (2 of 8)

Adding Kmer Bias (3 of 8)

Adding Kmer Bias (4 of 8)

Adding Kmer Bias (5 of 8)

Adding Kmer Bias (6 of 8)

Adding Kmer Bias (7 of 8)

Adding Kmer Bias (8 of 8)

2024-05-05 21:20:56 : Finished Creation of Coverage Files!, 1.264 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addGroupCoverages-53d3f42c958-Date-2024-05-05_Time-21-19-40.log

ArchR logging to : ArchRLogs/ArchR-addReproducibleP

                    Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
Fast-Firing   Fast-Firing    405        405           3   72  198   150000
Intermediate Intermediate    224        199           2   87  112    99500
Slow-Firing   Slow-Firing    389        389           3   68  207   150000


2024-05-05 21:20:57 : Batching Peak Calls!, 0.008 mins elapsed.

2024-05-05 21:20:57 : Batch Execution w/ safelapply!, 0 mins elapsed.

2024-05-05 21:22:42 : Identifying Reproducible Peaks!, 1.752 mins elapsed.

2024-05-05 21:22:48 : Creating Union Peak Set!, 1.85 mins elapsed.

Converged after 4 iterations!

Plotting Ggplot!

2024-05-05 21:22:52 : Finished Creating Union Peak Set (126587)!, 1.926 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-53d6310186c-Date-2024-05-05_Time-21-22-52.log
If there is an issue, please report to github with logFile!

2024-05-05 21:22:52 : Batch Execution w/ safelapply!, 0 mins elapsed.

Overriding previous entry for ReadsInPeaks

Overriding previous entry for FRIP

ArchR logging successful to : ArchRLogs/ArchR-addPeakMatrix-53d6310186c-Date-2024-05-05_Time-21-22-52.log



### Positive TF regulator input by alpha MN subtype

In [53]:
# Add motif annotations
proj_control_FF_Int_SF <- addMotifAnnotations(ArchRProj = proj_control_FF_Int_SF, motifSet = "cisbp", name = "Motif", force = TRUE)

ArchR logging to : ArchRLogs/ArchR-addMotifAnnotations-53d19bf764c-Date-2024-05-05_Time-21-26-30.log
If there is an issue, please report to github with logFile!

peakAnnotation name already exists! Overriding.

2024-05-05 21:26:30 : Gettting Motif Set, Species : Mus musculus, 0.005 mins elapsed.

Using version 2 motifs!

2024-05-05 21:26:34 : Finding Motif Positions with motifmatchr!, 0.071 mins elapsed.

2024-05-05 21:28:45 : All Motifs Overlap at least 1 peak!, 2.251 mins elapsed.

2024-05-05 21:28:45 : Creating Motif Overlap Matrix, 2.251 mins elapsed.

2024-05-05 21:28:47 : Finished Getting Motif Info!, 2.28 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addMotifAnnotations-53d19bf764c-Date-2024-05-05_Time-21-26-30.log



In [54]:
# Add background peaks
proj_control_FF_Int_SF <- addBgdPeaks(proj_control_FF_Int_SF)

Identifying Background Peaks!



In [55]:
# Add deviations matrix
proj_control_FF_Int_SF <- addDeviationsMatrix(
  ArchRProj = proj_control_FF_Int_SF, 
  peakAnnotation = "Motif",
  force = TRUE
)

Using Previous Background Peaks!

ArchR logging to : ArchRLogs/ArchR-addDeviationsMatrix-53d411cd275-Date-2024-05-05_Time-21-29-21.log
If there is an issue, please report to github with logFile!



NULL


'as(<lgCMatrix>, "dgCMatrix")' is deprecated.
Use 'as(., "dMatrix")' instead.
See help("Deprecated") and help("Matrix-deprecated").

2024-05-05 21:29:24 : Batch Execution w/ safelapply!, 0 mins elapsed.

###########
2024-05-05 21:32:48 : Completed Computing Deviations!, 3.457 mins elapsed.
###########

ArchR logging successful to : ArchRLogs/ArchR-addDeviationsMatrix-53d411cd275-Date-2024-05-05_Time-21-29-21.log



In [66]:
#Identify Deviant TF Motifs
seGroupMotif <- getGroupSE(ArchRProj = proj_control_FF_Int_SF, useMatrix = "MotifMatrix", groupBy = "alpha_subtype")

ArchR logging to : ArchRLogs/ArchR-getGroupSE-696f5364cc0e-Date-2024-08-09_Time-17-28-17.log
If there is an issue, please report to github with logFile!

Getting Group Matrix

2024-08-09 17:28:34 : Successfully Created Group Matrix, 0.038 mins elapsed.

Normalizing by number of Cells

ArchR logging successful to : ArchRLogs/ArchR-getGroupSE-696f5364cc0e-Date-2024-08-09_Time-17-28-17.log



In [67]:
seZ <- seGroupMotif[rowData(seGroupMotif)$seqnames=="z",]

In [68]:
rowData(seZ)$maxDelta <- lapply(seq_len(ncol(seZ)), function(x){
  rowMaxs(assay(seZ) - assay(seZ)[,x])
}) %>% Reduce("cbind", .) %>% rowMaxs

In [69]:
#Identify Correlated TF Motifs and TF Gene Score/Expression
corGEM_MM <- correlateMatrices(
    ArchRProj = proj_control_FF_Int_SF,
    useMatrix1 = "GeneExpressionMatrix",
    useMatrix2 = "MotifMatrix",
    reducedDims = "LSI_Combined"
)

ArchR logging to : ArchRLogs/ArchR-correlateMatrices-696f4ea1fb72-Date-2024-08-09_Time-17-28-35.log
If there is an issue, please report to github with logFile!

When accessing features from a matrix of class Sparse.Assays.Matrix it requires 1 seqname!
Continuing with first seqname 'z'!
If confused, try getFeatures(ArchRProj, 'MotifMatrix') to list out available seqnames for input!

2024-08-09 17:28:49 : Testing 797 Mappings!, 0.231 mins elapsed.

2024-08-09 17:28:49 : Computing KNN, 0.231 mins elapsed.

2024-08-09 17:28:50 : Identifying Non-Overlapping KNN pairs, 0.242 mins elapsed.

2024-08-09 17:28:51 : Identified 500 Groupings!, 0.266 mins elapsed.

2024-08-09 17:28:54 : Getting Group Matrix 1, 0.306 mins elapsed.

2024-08-09 17:29:00 : Getting Group Matrix 2, 0.418 mins elapsed.

Some entries in groupMat2 are less than 0, continuing without Log2 Normalization.
Most likely this assay is a deviations matrix.

Getting Correlations...

2024-08-09 17:29:03 : 

Computing Correlation (250

In [70]:
#Add Maximum Delta Deviation to the Correlation Data Frame
corGEM_MM$maxDelta <- rowData(seZ)[match(corGEM_MM$MotifMatrix_name, rowData(seZ)$name), "maxDelta"]

In [71]:
#Identify Positive TF Regulators
corGEM_MM <- corGEM_MM[order(abs(corGEM_MM$cor), decreasing = TRUE), ]
corGEM_MM <- corGEM_MM[which(!duplicated(gsub("\\-.*","",corGEM_MM[,"MotifMatrix_name"]))), ]
corGEM_MM$TFRegulator <- "NO"
corGEM_MM$TFRegulator[which(corGEM_MM$cor > 0.5 & corGEM_MM$padj < 0.01 & corGEM_MM$maxDelta > quantile(corGEM_MM$maxDelta, 0.75))] <- "YES"
sort(corGEM_MM[corGEM_MM$TFRegulator=="YES",1])

[1] "Arnt2"  "Arntl"  "Bach2"  "Creb1"  "Crem"   "Dlx2"   "Esrrb"  "Esrrg" 
 [9] "Etv6"   "Fos"    "Hoxc11" "Hoxd11" "Klf11"  "Klf13"  "Lhx4"   "Maf"   
[17] "Mitf"   "Nfib"   "Nr2f1"  "Nr2f2"  "Nr2f6"  "Nr4a1"  "Rora"   "Sp1"

In [72]:
write.csv(corGEM_MM, file='proj_control_FF_Int_SF_corGEM_MM.csv')